# Sim2 Image Perturbations

This notebook previews the Sim2 cropped dataset and visualises representative perturbations (noise and resolution degradations) used in the CLIP Monte Carlo Dropout study.

In [ ]:
from pathlib import Path
from typing import Callable, Dict, List, Optional

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120})
print("imported")

In [ ]:
# Resolve repository and dataset roots relative to the notebook location.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "data").exists():
    # When running from the notebooks/ directory, ascend one level to reach repo root.
    REPO_ROOT = REPO_ROOT.parent

DATA_ROOT = REPO_ROOT / "data/car_sim/sim2_cropped_45deg"
assert DATA_ROOT.exists(), f"Dataset not found at {DATA_ROOT}"

image_paths = sorted(DATA_ROOT.glob("*/*.png"))
print(f"Found {len(image_paths)} images")

In [ ]:
def load_images(paths: List[Path]) -> List[Image.Image]:
    images: List[Image.Image] = []
    for path in paths:
        with Image.open(path) as img:
            images.append(img.convert("RGB"))
    return images

images = load_images(image_paths)
len(images)

In [ ]:
def show_image_grid(images: List[Image.Image], titles: Optional[List[str]] = None, cols: int = 6, size: tuple = (18, 16)) -> None:
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=size)
    if rows == 1:
        axes = np.atleast_1d(axes)
    axes = axes.flatten()

    for ax, img, title in zip(axes, images, titles or ["" for _ in images]):
        ax.imshow(img)
        ax.axis("off")
        if title:
            ax.set_title(title, fontsize=9)

    # Blank remaining axes when the grid overflows.
    for ax in axes[len(images):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
original_titles = [f"{path.parent.name}\n{path.stem.split('_')[-1]}°" for path in image_paths]
show_image_grid(images, original_titles, cols=6, size=(20, 18))

In [ ]:
def gaussian_noise_transform(sigma: float, seed: int = 0) -> Callable[[Image.Image], Image.Image]:
    rng = np.random.default_rng(seed)

    def apply(image: Image.Image) -> Image.Image:
        array = np.asarray(image).astype(np.float32) / 255.0
        noise = rng.normal(loc=0.0, scale=sigma, size=array.shape)
        noisy = np.clip(array + noise, 0.0, 1.0)
        return Image.fromarray((noisy * 255.0).round().astype(np.uint8), mode=image.mode)

    return apply


def salt_pepper_transform(amount: float, salt_ratio: float = 0.5, seed: int = 0) -> Callable[[Image.Image], Image.Image]:
    rng = np.random.default_rng(seed)

    def apply(image: Image.Image) -> Image.Image:
        array = np.asarray(image).copy()
        h, w, _ = array.shape
        total_pixels = h * w
        noisy_pixels = int(amount * total_pixels)
        if noisy_pixels <= 0:
            return image
        salt_pixels = int(noisy_pixels * salt_ratio)
        pepper_pixels = noisy_pixels - salt_pixels
        indices = rng.choice(total_pixels, size=noisy_pixels, replace=False)
        salt_idx = indices[:salt_pixels]
        pepper_idx = indices[salt_pixels:]
        if salt_pixels > 0:
            ys, xs = np.divmod(salt_idx, w)
            array[ys, xs] = 255
        if pepper_pixels > 0:
            ys, xs = np.divmod(pepper_idx, w)
            array[ys, xs] = 0
        return Image.fromarray(array, mode=image.mode)

    return apply


def bicubic_downsample_transform(target_max_dim: int) -> Callable[[Image.Image], Image.Image]:
    def apply(image: Image.Image) -> Image.Image:
        width, height = image.size
        max_dim = max(width, height)
        if max_dim <= target_max_dim:
            return image
        scale = target_max_dim / float(max_dim)
        new_size = (max(1, int(round(width * scale))), max(1, int(round(height * scale))))
        coarse = image.resize(new_size, resample=Image.NEAREST)
        return coarse.resize((width, height), resample=Image.BICUBIC)

    return apply


def pixelate_transform(target_max_dim: int) -> Callable[[Image.Image], Image.Image]:
    def apply(image: Image.Image) -> Image.Image:
        width, height = image.size
        max_dim = max(width, height)
        if max_dim <= target_max_dim:
            return image
        scale = target_max_dim / float(max_dim)
        new_size = (max(1, int(round(width * scale))), max(1, int(round(height * scale))))
        coarse = image.resize(new_size, resample=Image.NEAREST)
        return coarse.resize((width, height), resample=Image.NEAREST)

    return apply


TRANSFORMS: Dict[str, Callable[[Image.Image], Image.Image]] = {
    "gaussian_sigma_0.30": gaussian_noise_transform(0.30, seed=42),
    "saltpepper_5pct": salt_pepper_transform(0.05, salt_ratio=0.5, seed=123),
    "downsample_bicubic_32px": bicubic_downsample_transform(32),
    "pixelate_32px": pixelate_transform(32),
}

list(TRANSFORMS.keys())

In [ ]:
transformed_sets: Dict[str, List[Image.Image]] = {}
for name, transform in TRANSFORMS.items():
    transformed_sets[name] = [transform(img) for img in images]

transformed_sets.keys()

In [ ]:
for name, imgs in transformed_sets.items():
    print(f"Transform: {name}")
    show_image_grid(imgs, original_titles, cols=6, size=(20, 18))